# Distribution & relationship plots: reading the workhorse charts

_Data Wrangling_

**A tour of the everyday EDA plots — what each one shows, and the trap hiding in each.**

Once your data is loaded and roughly clean, you look at it. EDA (Exploratory Data
       Analysis) is mostly plotting, and you keep reaching for the same dozen charts. This lesson is a
       field guide to them. We split the workhorses into two families by the question they answer.

       Distribution plots answer "what does one variable look like?" &mdash; where its
       values pile up, how spread out they are, whether there is one hump or several, where the
       outliers sit. The cast: histogram, KDE (Kernel Density Estimate), box plot,
       violin plot, and the ECDF (Empirical Cumulative Distribution Function).

---

This notebook is a practice scaffold for the **Distribution & relationship plots: reading the workhorse charts** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — matplotlib + seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine

sns.set_theme()

# Bundled real data: restaurant bills (one row per table).
tips = sns.load_dataset("tips")      # columns: total_bill, tip, day, time, size, ...

# ============================================================
# DISTRIBUTION PLOTS — "what does ONE variable look like?"
# ============================================================

# Histogram: bin the range and count. The bin count is the knob — try a few.
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, bins in zip(axes, [5, 20, 80]):
    sns.histplot(tips, x="total_bill", bins=bins, ax=ax)
    ax.set_title(f"histogram, {bins} bins")   # too few hides humps, too many invents spikes
plt.show()

# KDE: a smooth density. The bandwidth (bw_adjust) over/under-smooths.
sns.kdeplot(tips, x="total_bill", bw_adjust=0.3, label="narrow (wiggly)")
sns.kdeplot(tips, x="total_bill", bw_adjust=2.0, label="wide (over-smoothed)")
# cut=0 stops the curve spilling past the data's real min/max (bills can't be < 0).
sns.kdeplot(tips, x="total_bill", bw_adjust=1.0, cut=0, label="default, clipped")
plt.legend(); plt.show()

# Box plot: five-number summary per group — compact, but HIDES the shape.
sns.boxplot(tips, x="day", y="total_bill")
# overlay the raw points so a hidden second hump can't sneak past:
sns.stripplot(tips, x="day", y="total_bill", color="k", alpha=0.4, size=3)
plt.show()

# Violin: box + KDE, so the shape (e.g. bimodality) is visible.
sns.violinplot(tips, x="day", y="total_bill")
plt.show()

# ECDF: fraction of points <= x. No bins, no bandwidth — exact.
sns.ecdfplot(tips, x="total_bill")
plt.show()

# ============================================================
# RELATIONSHIP PLOTS — "how do TWO+ variables move together?"
# ============================================================

# Scatter: the default two-numeric view. alpha tames overplotting.
sns.scatterplot(tips, x="total_bill", y="tip", hue="time", alpha=0.5)
plt.show()
# For very large data, swap in a hexbin (a 2-D histogram):
# plt.hexbin(tips["total_bill"], tips["tip"], gridsize=25, cmap="Blues")

# Line: only when x is ordered (here, mean tip by party size — an ordered count).
order = tips.groupby("size")["tip"].mean().reset_index()
sns.lineplot(order, x="size", y="tip", marker="o")
plt.show()

# Heatmap: a correlation matrix of the wine features.
wine = load_wine(as_frame=True).frame
corr = wine.corr(numeric_only=True)
# diverging palette centered at 0, fixed -1..1 limits, annotated — an honest scale.
sns.heatmap(corr, vmin=-1, vmax=1, center=0, cmap="coolwarm")
plt.show()

# Pair plot: every feature vs every other (use only a handful of columns).
cols = ["alcohol", "flavanoids", "color_intensity", "proline", "target"]
sns.pairplot(wine[cols], hue="target", corner=True)
plt.show()

## Visualize the data & results

_The two main families on real wine data: a histogram of one feature (proline), and a scatter of two features (flavanoids vs color intensity) colored by grape class. What does each kind of plot reveal?_

In [ ]:
import numpy as np
from sklearn.datasets import load_wine

d = load_wine()
names = list(d.feature_names)
X, y = d.data, d.target

# --- DISTRIBUTION: histogram of one feature ('proline') ---
x = X[:, names.index('proline')]
edges = np.linspace(x.min(), x.max(), 9)          # 8 equal-width bins
counts = np.histogram(x, bins=edges)[0]
print('bin edges :', np.round(edges).astype(int)) # [278 453 628 804 979 1154 1330 1505 1680]
print('counts    :', counts)                      # [31 47 38 16 23 16  3  4]  (right-skewed)

# --- RELATIONSHIP: scatter of two features, colored by class ---
fx, fy = names.index('flavanoids'), names.index('color_intensity')
rng = np.random.RandomState(0)
for cls in [0, 1, 2]:
    idx = np.where(y == cls)[0]
    take = sorted(rng.choice(idx, 18, replace=False))      # 18 points per class
    pts = [[round(float(X[i, fx]), 2), round(float(X[i, fy]), 2)] for i in take]
    print(d.target_names[cls], pts)                        # classes separate cleanly

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A histogram of a "session length" column shows one smooth hump. A colleague insists there are really two kinds of users. How would you check, and which plots could have hidden the split?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Re-draw the histogram with several bin counts (e.g. 10, 30, 80). — _A single bin count can smear two nearby humps into one; sweeping bins reveals whether a gap appears._
- Overlay a KDE at a couple of bandwidths, and draw a violin or strip plot of the raw points. — _An over-smoothed KDE or a box plot would also collapse two humps into one shape; the violin/points show the true shape._
- Draw an ECDF and look for a flat shelf in the middle. — _A flat stretch in the ECDF means a region with no data — a gap between two clusters — shown exactly, with no tuning._

**Answer:** The smooth hump may be an artifact of too few bins or an over-smoothed KDE; a box plot would hide a split entirely. Check with several bin counts, a violin or strip plot of the points, and an ECDF (a flat shelf = a gap between two clusters). If a gap shows up, your colleague is right.

</details>

**Problem 2.** You scatter-plot 50,000 rows of (price, area) and see a solid black rectangle — no structure at all. List three ways to recover the density.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Lower the dot opacity with alpha (e.g. alpha=0.05). — _Where many points overlap the color builds up, so dense regions look darker and sparse ones lighter._
- Add small random jitter to any discrete/rounded coordinate. — _Identical values that would stack on one pixel get nudged apart, exposing how many there are._
- Switch to a hexbin or 2-D density plot. — _A hexbin is a 2-D histogram: it tiles the plane and colors each hexagon by its count, so density is encoded directly instead of by overlap._

**Answer:** This is overplotting. Recover the density by (1) lowering opacity (alpha), (2) adding jitter to discrete values, and (3) switching to a hexbin or 2-D density plot, which bins the plane and colors by count.

</details>

**Problem 3.** Why does an ECDF have no "knob" to mislead you, while a histogram and a KDE both do? Name the knob for each.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how each plot is built. — _A histogram bins the range; a KDE sums kernel bumps; an ECDF just counts the fraction of points at or below each value._
- Identify the tuning parameter of each. — _The histogram's knob is the bin count/width; the KDE's is the bandwidth $h$; the ECDF has none — it plots $F_n(x)$ directly._
- Connect each knob to a way it can mislead. — _Bad bins hide or invent humps; bad bandwidth over/under-smooths and spills past bounds; the ECDF, having no knob, cannot do either._

**Answer:** The histogram has a bin count/width and the KDE has a bandwidth $h$; either can hide real structure or invent fake structure. The ECDF is just $F_n(x)=$ the fraction of points $\le x$ — it has no bins and no bandwidth, so it shows the data exactly with nothing to tune.

</details>